# Séances 7–8 — Projet tutoré

**Decision problem:** what recommendation can we defend with evidence, limits, and a reproducible workflow?

Official topic preserved: guided project and restitution. The output is a Bloc 1 project brief.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]:
    d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"

def load_raw_trends():
    p = DATA / "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
    df = pd.read_csv(p)
    df["date"] = pd.to_datetime(df["date"])
    for c in ["chatgpt", "iphone", "meteo"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)

def load_clean_long():
    p = OUT / "bloc1" / "clean_trends_long.csv"
    if p.exists():
        df = pd.read_csv(p, parse_dates=["date"])
    else:
        raw = load_raw_trends()
        df = raw.melt(id_vars="date", var_name="signal", value_name="interest")
    return df.sort_values(["date", "signal"]).reset_index(drop=True)

In [ ]:
summary = pd.read_csv(OUT / "bloc1" / "eda_signal_summary.csv", index_col=0) if (OUT / "bloc1" / "eda_signal_summary.csv").exists() else pd.DataFrame()
metrics = pd.read_csv(OUT / "bloc1" / "model_metrics.csv") if (OUT / "bloc1" / "model_metrics.csv").exists() else pd.DataFrame()
brief = f"""# Bloc 1 Project Brief

Decision question: which public attention signal deserves monitoring for product decisions?

Key evidence:
{summary.to_string() if not summary.empty else 'Run S3 first.'}

Model check:
{metrics.to_string(index=False) if not metrics.empty else 'Run S4 first.'}

Recommendation: use the cleaned Google Trends pipeline as a lightweight weak-signal review, not as proof of demand.
"""
(OUT / "bloc1" / "project_brief.md").write_text(brief)
print(brief[:900])

## Practical exercise

Turn this brief into a 3-minute stakeholder presentation.

## Conclusion

The Bloc 1 project is now an evidence package feeding the rest of the data product.